# Territory Capture — All-Board Paper-Aligned Training (v2)

**Proje hedefi:** Kendi oyununu (Territory Capture) AlphaZero/AlphaStar mantığıyla AI ajan eğitmek. Multi-board (5×5, 6×6, 7×7) bu projenin extension'ı.

**Bu notebook (Seçenek A — Tam metodolojik tutarlılık):**
Üç board'u da **aynı paper-aligned recipe** ile yeniden eğitir. Tek değişen: model boyutu (her board'a göre channels/blocks). Bu sayede sonuçlar **apples-to-apples** karşılaştırılabilir — performans farkı kesinlikle board size'tan kaynaklanır.

## Eğitim Recipe (Paper-Aligned, Tüm Board'lar Aynı)

| Bileşen | Değer | Kaynak |
|---|---|---|
| Optimizer | Adam, lr=5e-4, wd=5e-4 | Paper §V-C |
| LR schedule | CosineAnnealingLR | Paper §V-C |
| Loss | Lπ + λv·Lv − λe·H(π) | Paper Eq. 4 |
| λv (value weight) | 0.1 | Paper Eq. 4 |
| λe (entropy weight) | 0.02 | Paper Eq. 4 |
| Epochs | 25 | Paper |
| Value head | FC(B²→32→1) | Paper §III-C |
| AMP + TF32 | Aktif | Paper §V-B |
| 8-fold augment | Aktif | Bizim eklemmiz |
| Early stopping | patience=6 | Bizim eklemmiz |

## Per-Board Mimari

| Board | Channels | Blocks | Value hidden | Dropout | Params (approx) | Batch |
|---|---|---|---|---|---|---|
| **5×5** | 64 | 5 | 32 | 0.10 | ~280k | 1024 |
| **6×6** | 64 | 5 | 32 | 0.10 | ~285k | 1024 |
| **7×7** | 96 | 8 | 32 | 0.10 | ~1.3M | 512 |

**Rationale:** 5×5 ve 6×6 küçük board, paper'ın 5-blok×64 yapısı yeter. 7×7 daha büyük state space, daha büyük model gerek.

## Eval Metrics (paper §VI-A)
- **Top-1 policy accuracy** (paper baseline: 78.7%)
- **Top-3 policy accuracy** (paper baseline: 83.6%)
- **Value MAE** (paper baseline: 0.229)

## 1. GPU + CUDA durumu

In [ ]:
!nvidia-smi -L
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Google Drive bağla + veri yükle

**Önce yerel makineden Drive'a şu dosyaları kopyalayın:**
- `colab/data_5x5.npz` (~20 MB)
- `colab/data_6x6.npz` (~30 MB)
- `colab/data_7x7.npz` (~40 MB)

Drive klasörü: `MyDrive/territory_capture/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/territory_capture'
import os
os.makedirs(DRIVE_BASE, exist_ok=True)
print('Drive:', sorted(os.listdir(DRIVE_BASE)))

In [ ]:
import shutil, os
for name in ['data_5x5.npz', 'data_6x6.npz', 'data_7x7.npz']:
    src = os.path.join(DRIVE_BASE, name)
    dst = f'/content/{name}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'  ✓ {name}: {os.path.getsize(dst)/1024/1024:.1f} MB')
    else:
        print(f'  ✗ {name} bulunamadı: {src}')

## 3. Model + Train kodu (inline, paper-aligned)

In [ ]:
%%writefile model_v2.py
import torch, torch.nn as nn, torch.nn.functional as F

class ResBlock(nn.Module):
    def __init__(self, ch, dp=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(ch)
        self.drop = nn.Dropout2d(p=dp) if dp > 0 else nn.Identity()
    def forward(self, x):
        r = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = self.drop(x)
        x += r
        return F.relu(x)

class PolicyValueNetV2(nn.Module):
    def __init__(self, board_size=6, in_channels=2, channels=64, num_blocks=5,
                 dropout_p=0.0, value_hidden=32):  # paper: value_hidden=32
        super().__init__()
        self.board_size = board_size
        A = board_size * board_size
        self.conv_in = nn.Conv2d(in_channels, channels, 3, padding=1, bias=False)
        self.bn_in = nn.BatchNorm2d(channels)
        self.res_blocks = nn.ModuleList([ResBlock(channels, dropout_p) for _ in range(num_blocks)])
        self.policy_conv = nn.Conv2d(channels, 2, 1, bias=False)
        self.policy_bn = nn.BatchNorm2d(2)
        self.policy_fc = nn.Linear(2 * A, A)
        self.policy_drop = nn.Dropout(p=dropout_p) if dropout_p > 0 else nn.Identity()
        self.value_conv = nn.Conv2d(channels, 1, 1, bias=False)
        self.value_bn = nn.BatchNorm2d(1)
        self.value_fc1 = nn.Linear(A, value_hidden)
        self.value_fc2 = nn.Linear(value_hidden, 1)
        self.value_drop = nn.Dropout(p=dropout_p) if dropout_p > 0 else nn.Identity()
    def forward(self, x):
        x = F.relu(self.bn_in(self.conv_in(x)))
        for b in self.res_blocks: x = b(x)
        p = F.relu(self.policy_bn(self.policy_conv(x))).view(x.size(0), -1)
        p = self.policy_fc(self.policy_drop(p))
        v = F.relu(self.value_bn(self.value_conv(x))).view(x.size(0), -1)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(self.value_drop(v)))
        return p, v

In [ ]:
%%writefile train_v2.py
import json
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader, Dataset, random_split
from model_v2 import PolicyValueNetV2

class NpzSelfPlayDataset(Dataset):
    def __init__(self, npz_path, augment=False):
        d = np.load(npz_path)
        self.states = d['states']; self.policies = d['policies']; self.values = d['values']
        self.board_size = int(d['board_size']); self.augment = augment; self.n = len(self.states)
    def __len__(self): return self.n
    def _sym(self, s, p, sid):
        B = self.board_size
        p2 = p.reshape(B, B)
        if sid >= 4: s = s[:, :, ::-1]; p2 = p2[:, ::-1]
        k = sid % 4
        if k: s = np.rot90(s, k=k, axes=(1, 2)); p2 = np.rot90(p2, k=k)
        return np.ascontiguousarray(s), np.ascontiguousarray(p2.reshape(-1))
    def __getitem__(self, i):
        s, p, v = self.states[i], self.policies[i], self.values[i]
        if self.augment:
            s, p = self._sym(s, p, np.random.randint(0, 8))
        return (torch.from_numpy(np.ascontiguousarray(s)),
                torch.from_numpy(np.ascontiguousarray(p)),
                torch.tensor([v], dtype=torch.float32))

def _topk(logits, target, k):
    return int((logits.topk(k, dim=1).indices == target.argmax(dim=1, keepdim=True)).any(dim=1).sum().item())

@dataclass
class Epoch:
    epoch: int; train_loss: float; train_policy: float; train_value: float; train_entropy: float
    val_loss: float; val_policy: float; val_value: float
    val_top1: float; val_top3: float; val_value_mae: float; lr: float

def _run(model, loader, dev, opt, train, vw, ew, scaler, amp):
    model.train(mode=train)
    tl = tp = tv = th = 0.0; n = 0
    top1 = top3 = 0; mae = 0.0; tot = 0
    for s, p, v in loader:
        s, p, v = s.to(dev, non_blocking=True), p.to(dev, non_blocking=True), v.to(dev, non_blocking=True)
        ctx = torch.amp.autocast(device_type=dev.type, enabled=amp)
        with torch.set_grad_enabled(train):
            with ctx:
                pl, pv = model(s)
                lp = F.log_softmax(pl, dim=1)
                pr = lp.exp()
                pol_loss = -(p * lp).sum(dim=1).mean()
                val_loss = F.mse_loss(pv, v)
                ent = -(pr * lp).sum(dim=1).mean()
                loss = pol_loss + vw * val_loss - ew * ent
            if train and opt is not None:
                opt.zero_grad(set_to_none=True)
                if scaler is not None and amp:
                    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                else:
                    loss.backward(); opt.step()
        with torch.no_grad():
            top1 += _topk(pl.float(), p, 1)
            top3 += _topk(pl.float(), p, 3)
            mae += float((pv.float() - v).abs().sum().item())
            tot += p.size(0)
        tl += loss.item(); tp += pol_loss.item(); tv += val_loss.item(); th += ent.item(); n += 1
    n = max(1, n); tot = max(1, tot)
    return dict(loss=tl/n, policy=tp/n, value=tv/n, entropy=th/n,
                top1=top1/tot, top3=top3/tot, value_mae=mae/tot)

def train(npz_path, output_path, board_size, channels=64, num_blocks=5, dropout_p=0.0,
          value_hidden=32, epochs=25, batch_size=1024, lr=5e-4, min_lr=1e-6, weight_decay=5e-4,
          value_weight=0.1, entropy_weight=0.02, val_split=0.05, augment=True, patience=6,
          num_workers=2, use_amp=True, use_tf32=True, device=None):
    dev = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    print(f'Device: {dev}', flush=True)
    if dev.type == 'cuda' and use_tf32:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
        print('TF32 + cudnn.benchmark enabled', flush=True)
    can_amp = (dev.type == 'cuda') and use_amp
    if can_amp: print('AMP enabled', flush=True)

    full = NpzSelfPlayDataset(npz_path, augment=augment)
    assert full.board_size == board_size
    n_val = max(1, int(len(full) * val_split)); n_tr = len(full) - n_val
    tr_set, va_set = random_split(full, [n_tr, n_val], generator=torch.Generator().manual_seed(42))
    tr_loader = DataLoader(tr_set, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                           pin_memory=(dev.type=='cuda'), persistent_workers=(num_workers>0))
    va_loader = DataLoader(va_set, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                           pin_memory=(dev.type=='cuda'), persistent_workers=(num_workers>0))

    model = PolicyValueNetV2(board_size=board_size, channels=channels, num_blocks=num_blocks,
                             dropout_p=dropout_p, value_hidden=value_hidden).to(dev)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'Model: ch={channels} blocks={num_blocks} dp={dropout_p} vh={value_hidden} params={n_params:,}', flush=True)

    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=min_lr)
    scaler = torch.amp.GradScaler() if can_amp else None

    history = []
    best_val = float('inf'); best_ep = 0; left = patience
    output = Path(output_path).resolve()
    output.parent.mkdir(parents=True, exist_ok=True)

    for ep in range(1, epochs+1):
        full.augment = augment
        tr = _run(model, tr_loader, dev, opt, True, value_weight, entropy_weight, scaler, can_amp)
        full.augment = False
        va = _run(model, va_loader, dev, None, False, value_weight, entropy_weight, None, can_amp)
        cur_lr = opt.param_groups[0]['lr']; sched.step()
        history.append(Epoch(ep, tr['loss'], tr['policy'], tr['value'], tr['entropy'],
                             va['loss'], va['policy'], va['value'],
                             va['top1'], va['top3'], va['value_mae'], cur_lr))
        print(f"Ep {ep}/{epochs} — train: {tr['loss']:.4f} (P:{tr['policy']:.4f} V:{tr['value']:.4f} H:{tr['entropy']:.3f}) "
              f"val: {va['loss']:.4f} (P:{va['policy']:.4f} V:{va['value']:.4f}) "
              f"top1: {va['top1']*100:.1f}% top3: {va['top3']*100:.1f}% MAE: {va['value_mae']:.3f} lr: {cur_lr:.6f}", flush=True)
        if va['loss'] < best_val - 1e-4:
            best_val = va['loss']; best_ep = ep; left = patience
            torch.save(model.state_dict(), output)
            print(f'  ✓ best saved (val={va["loss"]:.4f})', flush=True)
        else:
            left -= 1
            if left <= 0:
                print(f'Early stop @ epoch {ep}', flush=True); break

    meta = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'checkpoint_path': str(output),
        'board_size': board_size, 'channels': channels, 'num_blocks': num_blocks,
        'dropout_p': dropout_p, 'value_hidden': value_hidden,
        'value_weight': value_weight, 'entropy_weight': entropy_weight,
        'weight_decay': weight_decay, 'use_amp': can_amp, 'use_tf32': use_tf32 and dev.type == 'cuda',
        'epochs_run': len(history), 'best_epoch': best_ep, 'best_val_loss': best_val,
        'batch_size': batch_size, 'initial_lr': lr, 'min_lr': min_lr,
        'validation_split': val_split, 'augment': augment, 'patience': patience,
        'records_used': len(full),
        'paper_alignment': {
            'loss_form': 'Lπ + λv·Lv − λe·H(π)',
            'lambda_v': value_weight, 'lambda_e': entropy_weight,
            'optimizer': 'Adam', 'lr': lr, 'weight_decay': weight_decay,
            'scheduler': 'CosineAnnealingLR',
        },
        'history': [asdict(h) for h in history],
    }
    output.with_suffix('.metadata.json').write_text(json.dumps(meta, indent=2))
    print(f'\nDone. Best ep {best_ep} val={best_val:.4f}\nSaved: {output}', flush=True)
    return output

## 4. 5×5 eğit (makalenin ana board'u)

**Beklenen:** Paper baseline'ı yakala/aş — Top-1 ≈ %78, Top-3 ≈ %84, Value MAE ≈ 0.23.

In [ ]:
from train_v2 import train
out_5x5 = train(
    npz_path='/content/data_5x5.npz',
    output_path='/content/model_hard_5x5.pth',
    board_size=5,
    channels=64, num_blocks=5, dropout_p=0.10, value_hidden=32,
    epochs=25, batch_size=1024,
    lr=5e-4, weight_decay=5e-4,
    value_weight=0.1, entropy_weight=0.02,
    val_split=0.05, augment=True, patience=6,
    use_amp=True, use_tf32=True, num_workers=2,
)

In [ ]:
import shutil
shutil.copy('/content/model_hard_5x5.pth', f'{DRIVE_BASE}/model_hard_5x5.pth')
shutil.copy('/content/model_hard_5x5.metadata.json', f'{DRIVE_BASE}/model_hard_5x5.metadata.json')
print('5×5 modeli Drive\'a kaydedildi.')

## 5. 6×6 eğit (bizim extension'ımız — paper recipe ile)

Eski 6×6 modelimiz vardı (98% vs Random, 82% vs Heuristic, 56% vs Minimax) ama farklı recipe ile eğitilmişti (no augmentation, no entropy reg, value_weight=1.0). Bu sürüm üçünü de **aynı recipe**'ye getiriyor.

**Beklenen:** Eski 6×6 baseline'ından **eşit veya daha iyi** — augmentation + entropy reg + dropout iyileştirme getirmeli.

In [ ]:
out_6x6 = train(
    npz_path='/content/data_6x6.npz',
    output_path='/content/model_hard_6x6.pth',
    board_size=6,
    channels=64, num_blocks=5, dropout_p=0.10, value_hidden=32,
    epochs=25, batch_size=1024,
    lr=5e-4, weight_decay=5e-4,
    value_weight=0.1, entropy_weight=0.02,
    val_split=0.05, augment=True, patience=6,
    use_amp=True, use_tf32=True, num_workers=2,
)

In [ ]:
shutil.copy('/content/model_hard_6x6.pth', f'{DRIVE_BASE}/model_hard_6x6.pth')
shutil.copy('/content/model_hard_6x6.metadata.json', f'{DRIVE_BASE}/model_hard_6x6.metadata.json')
print('6×6 modeli Drive\'a kaydedildi.')

## 6. 7×7 eğit (paper'ın future work board'u)

**Konfig:** Daha büyük model (8 blok × 96 channel) — 7×7'nin geniş state space'i için.

Eski 7×7 modelimiz Heuristic'i bile yenememişti (Heuristic %100); bu sürüm augmentation + entropy reg ile daha iyi olmalı.

In [ ]:
out_7x7 = train(
    npz_path='/content/data_7x7.npz',
    output_path='/content/model_hard_7x7.pth',
    board_size=7,
    channels=96, num_blocks=8, dropout_p=0.10, value_hidden=32,
    epochs=25, batch_size=512,
    lr=5e-4, weight_decay=5e-4,
    value_weight=0.1, entropy_weight=0.02,
    val_split=0.05, augment=True, patience=6,
    use_amp=True, use_tf32=True, num_workers=2,
)

In [ ]:
shutil.copy('/content/model_hard_7x7.pth', f'{DRIVE_BASE}/model_hard_7x7.pth')
shutil.copy('/content/model_hard_7x7.metadata.json', f'{DRIVE_BASE}/model_hard_7x7.metadata.json')
print('7×7 modeli Drive\'a kaydedildi.')

## 7. (Opsiyonel) Doğrudan indirme

In [ ]:
from google.colab import files
for f in ['model_hard_5x5.pth', 'model_hard_5x5.metadata.json',
          'model_hard_6x6.pth', 'model_hard_6x6.metadata.json',
          'model_hard_7x7.pth', 'model_hard_7x7.metadata.json']:
    files.download(f'/content/{f}')

## 8. Yerel entegrasyon

`src/ai_agent.py` artık sibling `.metadata.json`'dan otomatik arch okuyor — indirilen modelleri `src/` altına koymak yeterli. Sonra:

```bash
# Yerelde, repo root'unda:
python -m experiments.tournament --mode both --boards 5 6 7 --games 50 \
    --sims-medium 30 --sims-hard 50 --include-baselines \
    --output results/tournament_final.json
python -m experiments.build_report --input results/tournament_final.json
```

**Beklenen toplam Colab süresi (T4 GPU):**
- 5×5: ~15 dk
- 6×6: ~20 dk
- 7×7: ~35 dk
- **Toplam: ~70 dk** (background eğitimi bekleyebilirsin)